# Streamlit 연동용 Qwen 감정 분석 데이터 가공 및 시각화 유틸리티

이 노트북은 Streamlit 개발 파트 담당자가 데이터를 쉽게 불러오고, 감정 수치 기반 상위 기사를 검색하며, 차트를 동적으로 그릴 수 있도록 돕는 공용 API 함수들을 제공합니다.
Streamlit 앱에서 이 모듈을 직접 import하여 한 줄 호출로 사용할 수 있습니다.

## 1단계: 필수 라이브러리 임포트 및 폰트 설정

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 한국어 폰트 설정 (Windows 기준 Malgun Gothic 적용)
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

## 2단계: 감정 분석 CSV 데이터 로드 함수

In [ ]:
def load_emotion_data(csv_path="data/News_Scraping_retouch_qwen.csv"):
    """
    Qwen 감정 분석 완료 CSV를 불러와 '날짜'를 datetime으로 변환하고
    '수치' 컬럼을 실수형으로 안전하게 보정하여 반환합니다.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"감정 분석 파일({csv_path})을 찾을 수 없습니다.")
    
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df['날짜'] = pd.to_datetime(df['날짜'])
    df['수치'] = pd.to_numeric(df['수치'], errors='coerce').fillna(0.0)
    return df

## 3단계: 시행 전/후 긍정·부정 여론 지표 계산 함수 (st.metric 연동용)

In [ ]:
def calculate_metrics(df, effective_date="2025-06-28"):
    """
    대책 시행일(effective_date)을 기준으로 시행 전과 후의 긍정/부정 비율 및
    그에 따른 증감폭(Delta)을 딕셔너리로 계산해 반환합니다.
    """
    eff_dt = pd.to_datetime(effective_date)
    
    before_df = df[df['날짜'] < eff_dt]
    after_df = df[df['날짜'] >= eff_dt]
    
    def get_ratios(sub_df):
        if len(sub_df) == 0:
            return {'긍정': 0.0, '부정': 0.0, '중립': 0.0}
        counts = sub_df['감정'].value_counts()
        total = len(sub_df)
        return {
            '긍정': (counts.get('긍정', 0) / total) * 100,
            '부정': (counts.get('부정', 0) / total) * 100,
            '중립': (counts.get('중립', 0) / total) * 100
        }
        
    before_ratios = get_ratios(before_df)
    after_ratios = get_ratios(after_df)
    
    delta_pos = after_ratios['긍정'] - before_ratios['긍정']
    delta_neg = after_ratios['부정'] - before_ratios['부정']
    
    return {
        'before_pos': before_ratios['긍정'],
        'after_pos': after_ratios['긍정'],
        'delta_pos': delta_pos,
        'before_neg': before_ratios['부정'],
        'after_neg': after_ratios['부정'],
        'delta_neg': delta_neg
    }

## 4단계: 특정 날짜의 감정 수치 상위 5순위 기사 조회 함수

In [ ]:
def get_top_news_by_date(df, target_date, top_n=5):
    """
    특정 날짜(target_date)에 감정의 신뢰도('수치')가 가장 높았던
    기사의 제목, URL, 감정 라벨, 수치를 내림차순 정렬하여 반환합니다.
    """
    target_dt = pd.to_datetime(target_date)
    df_day = df[df['날짜'].dt.date == target_dt.date()]
    
    if len(df_day) == 0:
        return pd.DataFrame()
        
    df_sorted = df_day.sort_values(by='수치', ascending=False).head(top_n)
    return df_sorted[['기사제목', 'url', '감정', '수치', '분류근거']]

## 5단계: 시간 흐름별 꺾은선 감정 추세 그래프 생성 함수

In [ ]:
def generate_trend_chart(df, interval_days=5, effective_date="2025-06-28"):
    """
    지정된 일수 단위(interval_days) 평균으로 집계하여
    감정별 뉴스 수 추이를 꺾은선 그래프(Matplotlib Figure)로 생성합니다.
    """
    df_daily = df.groupby(['날짜', '감정']).size().unstack(fill_value=0)
    for col in ['긍정', '중립', '부정']:
        if col not in df_daily.columns:
            df_daily[col] = 0
            
    all_dates = pd.date_range(start=df_daily.index.min(), end=df_daily.index.max(), freq='D')
    df_daily = df_daily.reindex(all_dates, fill_value=0)
    
    resample_rule = f"{interval_days}D"
    df_resampled = df_daily.resample(resample_rule).mean()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = {'부정': '#e74c3c', '중립': '#95a5a6', '긍정': '#2ecc71'}
    
    for col in ['부정', '중립', '긍정']:
        ax.plot(df_resampled.index, df_resampled[col], marker='o', linewidth=2, color=colors[col], label=col)
        
    eff_dt = pd.to_datetime(effective_date)
    ax.axvline(x=eff_dt, color='#3498db', linestyle='--', linewidth=2.5, label=f'시행일 ({effective_date})')
    
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=interval_days))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    
    ax.set_title(f'부동산 대책 시행 전후 감정 추이 ({interval_days}일 단위 일평균)', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('날짜', fontsize=10)
    ax.set_ylabel('일평균 뉴스 기사 수 (건)', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right', fontsize=10)
    plt.tight_layout()
    
    return fig

## 6단계: 시행 전/후 감정 비율 비교 막대 그래프 생성 함수

In [ ]:
def generate_comparison_bar_chart(df, effective_date="2025-06-28"):
    """
    시행일 전과 후의 긍정/중립/부정 점유비율(%)을 계산하여
    막대 차트(Matplotlib Figure)로 생성합니다.
    """
    df_temp = df.copy()
    eff_dt = pd.to_datetime(effective_date)
    
    df_temp['period'] = df_temp['날짜'].apply(lambda x: '시행 전' if x < eff_dt else '시행 후')
    
    pivot_df = df_temp.groupby(['period', '감정']).size().unstack(fill_value=0)
    for col in ['긍정', '중립', '부정']:
        if col not in pivot_df.columns:
            pivot_df[col] = 0
            
    ratio_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100
    ratio_df = ratio_df.reindex(columns=['긍정', '중립', '부정'])
    
    fig, ax = plt.subplots(figsize=(8, 4.8))
    colors = ['#2ecc71', '#95a5a6', '#e74c3c']
    
    ratio_df.plot(kind='bar', stacked=False, color=colors, ax=ax, width=0.6)
    
    ax.set_title('대책 시행 전 vs 후 감정 비율 (%) 비교', fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('비율 (%)', fontsize=10)
    ax.set_xlabel('시행 여부', fontsize=10)
    plt.xticks(rotation=0)
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    ax.legend(title='감정 라벨', loc='upper right')
    
    # 바 상단에 소수점 첫째자리 비율 표기
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.1f}%",
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                        textcoords='offset points')
            
    plt.tight_layout()
    return fig

## 7단계: 주피터 노트북 실행 예시 및 설명 (전체 주석 처리)

In [ ]:
# ==========================================================================
# [Streamlit 실시간 연동 예시 및 사용 방법 설명]
# ==========================================================================
#
# 이 노트북 내 정의된 함수들은 Streamlit 연동 시 백엔드 API 역할을 수행합니다.
# Streamlit 담당자는 아래 예시 주석과 같이 st 함수들을 사용해 웹 앱에 바로 렌더링할 수 있습니다.
#
# --------------------------------------------------------------------------
# 1. 데이터 불러오기
# --------------------------------------------------------------------------
# import streamlit as st
#
# try:
#     # 분석 완료된 데이터를 읽어옵니다.
#     df = load_emotion_data("data/News_Scraping_retouch_qwen.csv")
#     st.success(f"데이터 로드 성공! 기사 개수: {len(df)}개")
# except Exception as e:
#     st.error(f"데이터 로드 실패: {e}")
#
# --------------------------------------------------------------------------
# 2. 주요 변화 지표(st.metric) 렌더링 예시
# --------------------------------------------------------------------------
# # 대책 시행일을 기준으로 시행 전/후 비율 및 변화량 계산
# metrics = calculate_metrics(df, effective_date="2025-06-28")
#
# col1, col2 = st.columns(2)
# with col1:
#     st.metric(
#         label="시행 후 긍정 비율", 
#         value=f"{metrics['after_pos']:.2f}%", 
#         delta=f"{metrics['delta_pos']:.2f}%p"
#     )
# with col2:
#     st.metric(
#         label="시행 후 부정 비율", 
#         value=f"{metrics['after_neg']:.2f}%", 
#         delta=f"{metrics['delta_neg']:.2f}%p",
#         delta_color="inverse" # 부정 증가 시 붉은 경고색으로 표시
#     )
#
# --------------------------------------------------------------------------
# 3. 날짜 조절(st.select_slider) 및 당일 감정 수치 상위 5순위 기사 조회 예시
# --------------------------------------------------------------------------
# # 사용자가 드래그(Slider)하여 날짜를 선택할 수 있게 바 생성
# import datetime
# min_date = df['날짜'].min().date()
# max_date = df['날짜'].max().date()
#
# selected_date = st.select_slider(
#     "조회할 날짜를 선택하세요:",
#     options=pd.date_range(start=min_date, end=max_date, freq='D').date,
#     value=min_date
# )
#
# # 해당 날짜의 감정 신뢰 수치('수치')가 가장 높은 상위 5건 검색
# top_news = get_top_news_by_date(df, target_date=selected_date, top_n=5)
#
# st.subheader(f"📅 {selected_date} 감정 신뢰도 상위 5순위 뉴스")
# if top_news.empty:
#     st.info("해당 날짜에 수집 및 분석된 뉴스 기사가 없습니다.")
# else:
#     for i, row in enumerate(top_news.itertuples(), 1):
#         # 감정에 따른 텍스트 뱃지 색상 매핑
#         color = "green" if row.감정 == "긍정" else ("red" if row.감정 == "부정" else "gray")
#         
#         st.markdown(f"**{i}위.** [{row.기사제목}]({row.url})")
#         st.markdown(f"* 감정분류: :{color}[{row.감정}] (신뢰도 수치: `{row.수치:.4f}`)")
#         st.markdown(f"* 분류근거: *{row.분류근거}*")
#         st.markdown("---")
#
# --------------------------------------------------------------------------
# 4. 차트 동적 생성 및 화면 출력 예시 (st.pyplot 연동)
# --------------------------------------------------------------------------
# # 꺾은선 그래프 Figure 객체를 리턴받아 웹 화면에 실시간 렌더링
# fig_trend = generate_trend_chart(df, interval_days=5, effective_date="2025-06-28")
# st.pyplot(fig_trend)
#
# # 비율 비교 막대 그래프 Figure 객체를 리턴받아 화면에 렌더링
# fig_bar = generate_comparison_bar_chart(df, effective_date="2025-06-28")
# st.pyplot(fig_bar)
# ==========================================================================